In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, median_absolute_error, r2_score

## Import

In [12]:
X = pd.read_csv('X_trainval_preprocessed.csv')
y = pd.read_csv('y_trainval.csv').values.ravel()

In [13]:
X.head()

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,mileage_per_year,tax_engineSize,age_mileage,...,model_corsa,model_i3,model_ka+,transmission_Manual,transmission_Semi-Auto,transmission_Unknown,fuelType_Hybrid,fuelType_Other,fuelType_Petrol,fuelType_Unknown
0,0.851852,0.087988,0.25,0.021966,0.303030,0.500000,0.666667,0.018668,0.058398,0.046582,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.907407,0.014204,0.25,0.099638,0.227273,0.395161,0.166667,0.004521,0.060484,0.005013,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.907407,0.011217,0.25,0.084735,0.227273,0.443548,0.666667,0.003570,0.060484,0.003959,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,0.888889,0.028177,0.25,0.137535,0.151515,0.395161,0.333333,0.007686,0.040323,0.011602,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.907407,0.003093,0.25,0.088780,0.227273,0.774194,0.500000,0.000985,0.060484,0.001092,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


## Cross-Validation

In [14]:
#Using the Standard Cross-Validation to evaluate the model
cv = KFold(n_splits=5, shuffle=True, random_state=42)

## Models 

In [15]:
#Models to use
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1),
    'Random Forest': RandomForestRegressor(n_estimators= 100, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
}

In [16]:
#Metrics to use
metrics = {
    'R2': 'r2',
    'MAE': 'neg_mean_absolute_error',
    'MSE': 'neg_mean_squared_error',
    'MAPE': 'neg_mean_absolute_percentage_error',
    'MedAE': 'neg_median_absolute_error',
}

In [17]:
results = {}

for name, model in models.items():
    cv_results = cross_validate(model, X, y, cv=cv, scoring=metrics, n_jobs=-1)
    # Store the mean of each metric (invert sign for errors)
    results[name] = {
        'R2': cv_results['test_R2'].mean(),
        'MAE': -cv_results['test_MAE'].mean(),
        'MSE': -cv_results['test_MSE'].mean(),
        'MAPE': -cv_results['test_MAPE'].mean(),
        'MedAE': -cv_results['test_MedAE'].mean()
    }

In [18]:
#Results for each model
metrics_table = pd.DataFrame(results).T

metrics_table.round(3)

,R2,MAE,MSE,MAPE,MedAE
Linear Regression,0.833,2441.307,1.587700e+07,0.172,1655.034
Ridge,0.833,2448.449,1.587421e+07,0.172,1661.296
Random Forest,0.931,1458.488,6.568065e+06,0.092,891.803
Decision Tree,0.884,1923.768,1.103020e+07,0.121,1104.200


## Best Model

In [19]:
best_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
best_model.fit(X,y)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [20]:
#Use Random Forest to predict our target
X_test = pd.read_csv('X_test_preprocessed.csv')
y_pred = best_model.predict(X_test)

## Kaggle

In [21]:
#Create the dataframe to submit into the Kaggle competition
kaggle_submission = pd.DataFrame({
    'CarID': pd.read_csv('project_data/test.csv')['carID'],
    'price': y_pred
})

In [22]:
kaggle_submission.to_csv('kaggle_submission.csv', index=False)